In [1]:
import pickle 
import numpy as np 
import itertools
import pandas as pd
from pathlib import Path 


# Generate conditions for symmetric distractor test.

Test in style of Kidd / Marrone et al. (2008) / Srinivasan et al. (2016) experiments 

Target at 0 aziuth    
Distractors symmetrically presented at +/- 0, 5, 10, 20, and 40 degrees azimuth    
All signals at 0 elevation     
Left and right distractor level summed and set to 60dB SPL
- equal power at left and right for model first pass
- human experiments roved L/R balance slightly

Target set against distractor at desired SNR 

### Want to test effect of reverberation in this setting 
Use rooms at `/om2/user/msaddler/spatial_audio_pipeline/assets/brir/eval/` which are versions of the speaker array room with alternative reverb profiles.

## This Notebook
Generate location x snr conditions to get model performance and thresholds, per room 

## Make manifests for new minimally reverberant simulation of MIT 46 1004

### Load manifest of rooms

In [2]:
new_min_rev_dir = Path("/om2/user/imgriff/spatial_audio_pipeline/assets/brir/mit_bldg46room1004_min_reverb")
room_manifest = pd.read_pickle(new_min_rev_dir / 'manifest_room.pdpkl')
room_manifest

,head_azim,head_pos_xyz,index_room,is_outdoor,material_x0,material_x1,material_y0,material_y1,material_z0,material_z1,room_dim_xyz,room_materials
0,0,"[2.3, 3.6, 0.9]",0,False,wood panelling on glass fiber blanket,wood panelling on glass fiber blanket,wood panelling on glass fiber blanket,wood panelling on glass fiber blanket,carpet on foam rubber padding,"highly absorptive panels, 1in, 16in below ceiling","[4.66, 5.9, 2.48]","[11, 11, 11, 11, 15, 20]"
1,0,"[3.6, 2.36, 0.9]",1,False,wood panelling on glass fiber blanket,wood panelling on glass fiber blanket,wood panelling on glass fiber blanket,wood panelling on glass fiber blanket,carpet on foam rubber padding,"highly absorptive panels, 1in, 16in below ceiling","[5.9, 4.66, 2.48]","[11, 11, 11, 11, 15, 20]"
2,0,"[2.36, 2.3, 0.9]",2,False,wood panelling on glass fiber blanket,wood panelling on glass fiber blanket,wood panelling on glass fiber blanket,wood panelling on glass fiber blanket,carpet on foam rubber padding,"highly absorptive panels, 1in, 16in below ceiling","[4.66, 5.9, 2.48]","[11, 11, 11, 11, 15, 20]"
3,0,"[2.3, 2.3, 0.9]",3,False,wood panelling on glass fiber blanket,wood panelling on glass fiber blanket,wood panelling on glass fiber blanket,wood panelling on glass fiber blanket,carpet on foam rubber padding,"highly absorptive panels, 1in, 16in below ceiling","[5.9, 4.66, 2.48]","[11, 11, 11, 11, 15, 20]"


#### Manifest to re-run azimuth thresholds in new room

In [9]:

target_azim = 0
elevations = [-20, 0, 40]

# distractor_azims = [0, 10, 30] 
distractor_azims = [0, 5, 10, 20, 30, 40, 60, 90, 120, 150, 180] # Add 30, 60, 90 for comparison to our human experiments - test script handles l/r positioning 


distractor_elevation_deltas = [0, 5, 20, 40]

snrs = np.linspace(6,-21,num=10)
room_ixs = [0]

new_min_rev_dir = Path("/om2/user/imgriff/spatial_audio_pipeline/assets/brir/mit_bldg46room1004_min_reverb")
room_manifest = pd.read_pickle(new_min_rev_dir / 'manifest_room.pdpkl')

def get_room_str(room_ix):
    return  (new_min_rev_dir / f"room{room_ix:04}.hdf5").as_posix()

def get_room_meta_dict(room_ix):
    manifest_path = (new_min_rev_dir / "manifest_brir.pdpkl").as_posix()
    index_room = room_ix
    h5_fn = get_room_str(room_ix)
    return dict(room_manifest=manifest_path, index_room=index_room, h5_fn=h5_fn)

# create dict of trial dicts for azimuth experiment 

azim_cond_dict = {ix:{'target_loc': (0, elev),
                      'distract_loc': [dist_az, elev],
                      'snr': snr,
                      'symmetric_distractor': True,
                      'test_room_meta': get_room_meta_dict(room_ix)} for ix, (elev, dist_az, snr, room_ix) in
              enumerate(itertools.product(*[elevations, distractor_azims, snrs, room_ixs]))} 

n_azim_conds = len(azim_cond_dict)
# create dict of trial dicts for elevation experiment 
elev_cond_dict = {n_azim_conds + ix :{'target_loc': (0, elev),
                      'distract_loc': [0, elev - (np.sign(elev) * elev_delta)],
                      'snr': snr,
                      'symmetric_distractor': False,
                      'test_room_meta': get_room_meta_dict(room_ix)} for ix, (elev, elev_delta, snr, room_ix) in
              enumerate(itertools.product(*[elevations, distractor_elevation_deltas, snrs, room_ixs]))} 


all_cond_dict = {**azim_cond_dict, **elev_cond_dict}
# Assert keys contiguous
assert np.all(np.array(list(all_cond_dict.keys())) == np.arange(len(all_cond_dict)))
print(len(all_cond_dict))

450


### Add anechoic room 

In [10]:

target_azim = 0
elevations = [-20, 0, 40]

# distractor_azims = [0, 10, 30] 
distractor_azims = [0, 5, 10, 20, 30, 40, 60, 90, 120, 150, 180] # Add 30, 60, 90 for comparison to our human experiments - test script handles l/r positioning 


distractor_elevation_deltas = [0, 5, 20, 40]

snrs = np.linspace(6,-21,num=10)
room_ixs = [0]

def get_room_str(room_ix):
    return f"/om2/user/msaddler/spatial_audio_pipeline/assets/brir/eval/room{room_ix:04}.hdf5"

def get_room_meta_dict(room_ix):
    manifest_path = "/om2/user/msaddler/spatial_audio_pipeline/assets/brir/eval/manifest_brir.pdpkl"
    index_room = room_ix
    h5_fn = get_room_str(room_ix)
    return dict(room_manifest=manifest_path, index_room=index_room, h5_fn=h5_fn)


# create dict of trial dicts for azimuth experiment 
n_existing = len(all_cond_dict)
azim_cond_dict = {ix + n_existing :{'target_loc': (0, elev),
                      'distract_loc': [dist_az, elev],
                      'snr': snr,
                      'symmetric_distractor': True,
                      'test_room_meta': get_room_meta_dict(room_ix)} for ix, (elev, dist_az, snr, room_ix) in
              enumerate(itertools.product(*[elevations, distractor_azims, snrs, room_ixs]))} 

n_azim_conds = len(azim_cond_dict) + n_existing
# create dict of trial dicts for elevation experiment 
elev_cond_dict = {n_azim_conds + ix :{'target_loc': (0, elev),
                      'distract_loc': [0, elev - (np.sign(elev) * elev_delta)],
                      'snr': snr,
                      'symmetric_distractor': False,
                      'test_room_meta': get_room_meta_dict(room_ix)} for ix, (elev, elev_delta, snr, room_ix) in
              enumerate(itertools.product(*[elevations, distractor_elevation_deltas, snrs, room_ixs]))} 


all_cond_dict = {**all_cond_dict, **azim_cond_dict, **elev_cond_dict}
# Assert keys contiguous
assert np.all(np.array(list(all_cond_dict.keys())) == np.arange(len(all_cond_dict)))
print(len(all_cond_dict))

900


### Add standard speaker array

In [11]:

target_azim = 0
elevations = [-20, 0, 40]

# distractor_azims = [0, 10, 30] 
distractor_azims = [0, 5, 10, 20, 30, 40, 60, 90, 120, 150, 180] # Add 30, 60, 90 for comparison to our human experiments - test script handles l/r positioning 


distractor_elevation_deltas = [0, 5, 20, 40]

snrs = np.linspace(6,-21,num=10)
room_ixs = [0]

def get_room_str(room_ix):
    return f"/om2/user/msaddler/spatial_audio_pipeline/assets/brir/mit_bldg46room1004/room{room_ix:04}.hdf5"

def get_room_meta_dict(room_ix):
    manifest_path = "/om2/user/msaddler/spatial_audio_pipeline/assets/brir/mit_bldg46room1004/manifest_brir.pdpkl"
    index_room = room_ix
    h5_fn = get_room_str(room_ix)
    return dict(room_manifest=manifest_path, index_room=index_room, h5_fn=h5_fn)


# create dict of trial dicts for azimuth experiment 
n_existing = len(all_cond_dict)
azim_cond_dict = {ix + n_existing :{'target_loc': (0, elev),
                      'distract_loc': [dist_az, elev],
                      'snr': snr,
                      'symmetric_distractor': True,
                      'test_room_meta': get_room_meta_dict(room_ix)} for ix, (elev, dist_az, snr, room_ix) in
              enumerate(itertools.product(*[elevations, distractor_azims, snrs, room_ixs]))} 

n_azim_conds = len(azim_cond_dict) + n_existing
# create dict of trial dicts for elevation experiment 
elev_cond_dict = {n_azim_conds + ix :{'target_loc': (0, elev),
                      'distract_loc': [0, elev - (np.sign(elev) * elev_delta)],
                      'snr': snr,
                      'symmetric_distractor': False,
                      'test_room_meta': get_room_meta_dict(room_ix)} for ix, (elev, elev_delta, snr, room_ix) in
              enumerate(itertools.product(*[elevations, distractor_elevation_deltas, snrs, room_ixs]))} 


all_cond_dict = {**all_cond_dict, **azim_cond_dict, **elev_cond_dict}
# Assert keys contiguous
assert np.all(np.array(list(all_cond_dict.keys())) == np.arange(len(all_cond_dict)))
print(len(all_cond_dict))

1350


In [12]:
len(all_cond_dict)

1350

In [11]:
# all_conds
# all conditions 
# with open('binaural_test_manifests/symmetric_distractor_conditions_w_front_back_neg_21_to_6_dBSNR_min_reverb_mit_room.pkl', 'wb') as f:
with open('binaural_test_manifests/symmetric_distractor_conditions_w_front_back_neg_21_to_6_dBSNR_test_rooms.pkl', 'wb') as f:
    pickle.dump(all_cond_dict, f)


## Alternate rooms 

### Load manifest of rooms

In [2]:
room_manifest = pd.read_pickle('/om2/user/msaddler/spatial_audio_pipeline/assets/brir/eval/manifest_room.pdpkl')

In [3]:
room_manifest

,head_azim,head_pos_xyz,index_room,is_outdoor,material_x0,material_x1,material_y0,material_y1,material_z0,material_z1,room_dim_xyz,room_materials
0,0,"(2.5, 2.5, 2.0)",0,False,"(Anechoic,)","(Anechoic,)","(Anechoic,)","(Anechoic,)","(Anechoic,)","(Anechoic,)","(5.0, 5.0, 4.0)","(26, 26, 26, 26, 26, 26)"
1,0,"(2.5, 2.5, 2.0)",1,False,"(Wood panelling on glass fiber blanket,)","(Wood panelling on glass fiber blanket,)","(Wood panelling on glass fiber blanket,)","(Wood panelling on glass fiber blanket,)","(Carpet on foam rubber padding,)","(Highly absorptive panels, 1"", 16"" below ceili...","(5.0, 5.0, 4.0)","(11, 11, 11, 11, 15, 20)"
2,0,"(2.5, 2.5, 2.0)",2,False,"(Brick,)","(Brick,)","(Brick,)","(Brick,)","(Wood parquet on concrete,)","(Plaster, gypsum, or lime on lath,)","(5.0, 5.0, 4.0)","(1, 1, 1, 1, 12, 16)"
3,0,"(2.5, 2.5, 2.0)",3,False,"(Fiberglass wall treatment, 1 in,)","(Fiberglass wall treatment, 1 in,)","(Fiberglass wall treatment, 1 in,)","(Fiberglass wall treatment, 1 in,)","(Linoleum,)","(Acoustic tiles, 0.625"", 16"" below ceiling,)","(5.0, 5.0, 4.0)","(9, 9, 9, 9, 13, 17)"
4,0,"(2.5, 2.5, 2.0)",4,False,"(Concrete, painted,)","(Concrete, painted,)","(Concrete, painted,)","(Concrete, painted,)","(Linoleum,)","(Acoustic tiles, 0.625"", 16"" below ceiling,)","(5.0, 5.0, 4.0)","(2, 2, 2, 2, 13, 17)"
5,0,"(2.3, 3.6, 0.9)",5,False,"(Fiberglass wall treatment, 1 in,)","(Fiberglass wall treatment, 1 in,)","(Fiberglass wall treatment, 1 in,)","(Concrete, painted,)","(Linoleum,)","(Acoustic tiles, 0.625"", 16"" below ceiling,)","(4.66, 5.9, 2.48)","(9, 9, 9, 2, 13, 17)"
6,0,"(2.33, 2.95, 0.9)",6,False,"(Fiberglass wall treatment, 1 in,)","(Fiberglass wall treatment, 1 in,)","(Fiberglass wall treatment, 1 in,)","(Concrete, painted,)","(Linoleum,)","(Acoustic tiles, 0.625"", 16"" below ceiling,)","(4.66, 5.9, 2.48)","(9, 9, 9, 2, 13, 17)"
7,0,"(2.3, 3.6, 0.9)",7,False,"(Fiberglass wall treatment, 1 in,)","(Fiberglass wall treatment, 1 in,)","(Fiberglass wall treatment, 1 in,)","(Fiberglass wall treatment, 1 in,)","(Linoleum,)","(Acoustic tiles, 0.625"", 16"" below ceiling,)","(4.66, 5.9, 2.48)","(9, 9, 9, 9, 13, 17)"
8,0,"(2.33, 2.95, 0.9)",8,False,"(Fiberglass wall treatment, 1 in,)","(Fiberglass wall treatment, 1 in,)","(Fiberglass wall treatment, 1 in,)","(Fiberglass wall treatment, 1 in,)","(Linoleum,)","(Acoustic tiles, 0.625"", 16"" below ceiling,)","(4.66, 5.9, 2.48)","(9, 9, 9, 9, 13, 17)"


In [4]:
room_ir_df = pd.read_pickle('/om2/user/msaddler/spatial_audio_pipeline/assets/brir/eval/manifest_brir.pdpkl')
elevations = room_ir_df[(room_ir_df.src_dist == 1.4) & (room_ir_df.index_room == 0)].src_elev.unique() # np.arange(-20, 41, 10)
azimuths = room_ir_df[(room_ir_df.src_dist == 1.4) & (room_ir_df.index_room == 0)].src_azim.unique() # np.arange(-20, 41, 10)
# remove 0 elevation
print(azimuths)
print(elevations)


[  0   5  10  15  20  25  30  35  40  45  50  55  60  65  70  75  80  85
  90  95 100 105 110 115 120 125 130 135 140 145 150 155 160 165 170 175
 180 185 190 195 200 205 210 215 220 225 230 235 240 245 250 255 260 265
 270 275 280 285 290 295 300 305 310 315 320 325 330 335 340 345 350 355]
[-40 -30 -20 -10   0  10  20  30  40  50  60]


In [5]:
room_ir_df[room_ir_df.index_room == 0]

,buffer,c,dur,head_azim,head_pos_xyz,incorporate_lead_zeros,index_brir,index_room,room_dim_xyz,room_materials,sr,src_azim,src_dist,src_elev,use_highpass,use_hrtf_symmetry,use_jitter,use_log_distance
0,0,344.5,0.5,0,"(2.5, 2.5, 2.0)",True,0,0,"(5.0, 5.0, 4.0)","(26, 26, 26, 26, 26, 26)",44100,0,1.4,-40,True,True,True,False
1,0,344.5,0.5,0,"(2.5, 2.5, 2.0)",True,1,0,"(5.0, 5.0, 4.0)","(26, 26, 26, 26, 26, 26)",44100,0,1.4,-30,True,True,True,False
2,0,344.5,0.5,0,"(2.5, 2.5, 2.0)",True,2,0,"(5.0, 5.0, 4.0)","(26, 26, 26, 26, 26, 26)",44100,0,1.4,-20,True,True,True,False
3,0,344.5,0.5,0,"(2.5, 2.5, 2.0)",True,3,0,"(5.0, 5.0, 4.0)","(26, 26, 26, 26, 26, 26)",44100,0,1.4,-10,True,True,True,False
4,0,344.5,0.5,0,"(2.5, 2.5, 2.0)",True,4,0,"(5.0, 5.0, 4.0)","(26, 26, 26, 26, 26, 26)",44100,0,1.4,0,True,True,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1579,0,344.5,0.5,0,"(2.5, 2.5, 2.0)",True,1579,0,"(5.0, 5.0, 4.0)","(26, 26, 26, 26, 26, 26)",44100,355,2.0,20,True,True,True,False
1580,0,344.5,0.5,0,"(2.5, 2.5, 2.0)",True,1580,0,"(5.0, 5.0, 4.0)","(26, 26, 26, 26, 26, 26)",44100,355,2.0,30,True,True,True,False
1581,0,344.5,0.5,0,"(2.5, 2.5, 2.0)",True,1581,0,"(5.0, 5.0, 4.0)","(26, 26, 26, 26, 26, 26)",44100,355,2.0,40,True,True,True,False
1582,0,344.5,0.5,0,"(2.5, 2.5, 2.0)",True,1582,0,"(5.0, 5.0, 4.0)","(26, 26, 26, 26, 26, 26)",44100,355,2.0,50,True,True,True,False


### Get range of conditions that match human tests (ours and published)

In [6]:
np.linspace(6,-12,num=7)

array([  6.,   3.,   0.,  -3.,  -6.,  -9., -12.])

In [12]:
# First add conditions for confusion matrix 
target_azim = 0
elevation = 0 
distractor_azims = [0, 5, 10, 20, 30, 40, 60, 90] # Add 30, 60, 90 for comparison to our human experiments - test script handles l/r positioning 
snrs = np.linspace(6,-15,num=8)
print(snrs)
room_ixs = room_manifest.index_room.unique()
print(room_ixs)

def get_room_str(room_ix):
    return f"/om2/user/msaddler/spatial_audio_pipeline/assets/brir/eval/room{room_ix:04}.hdf5"

def get_room_meta_dict(room_ix):
    manifest_path = "/om2/user/msaddler/spatial_audio_pipeline/assets/brir/eval/manifest_brir.pdpkl"
    index_room = room_ix
    h5_fn = get_room_str(room_ix)
    return dict(room_manifest=manifest_path, index_room=index_room, h5_fn=h5_fn)

# create dict of trial dicts
all_cond_dict = {ix:{'target_loc': (target_azim, elevation),
                      'distract_loc': [dist_az, elevation],
                      'snr': snr,
                      'test_room_meta': get_room_meta_dict(room_ix)} for ix, (dist_az, snr, room_ix) in
              enumerate(itertools.product(*[distractor_azims, snrs, room_ixs]))} 

n_all_cond_dict = len(all_cond_dict)
print(n_all_cond_dict)

# Assert keys contiguous
assert np.all(np.array(list(all_cond_dict.keys())) == np.arange(len(all_cond_dict)))

[  6.   3.   0.  -3.  -6.  -9. -12. -15.]
[0 1 2 3 4 5 6 7 8]
576


In [16]:
# all_cond_dict

In [13]:
# all_conds
# all conditions 
with open('binaural_test_manifests/symmetric_distractor_conditions_neg_15_to_6_dBSNR_eval_rooms.pkl', 'wb') as f:
    pickle.dump(all_cond_dict, f)

### Make conditions for single-distractor thresholds measuring 0 to 180 azimuth separations 

In [3]:
# First add conditions for confusion matrix 
target_azim = 0
elevation = 0 
distractor_azims = np.arange(0,181,10) # [0, 10, 20, 30, 40, 60, 90] # Add 30, 60, 90 for comparison to our human experiments - test script handles l/r positioning 
snrs = np.linspace(6,-21,num=10)
print(snrs)
room_ixs = [0]# only want anechoic room 
print(room_ixs)

def get_room_str(room_ix):
    return f"/om2/user/msaddler/spatial_audio_pipeline/assets/brir/eval/room{room_ix:04}.hdf5"

def get_room_meta_dict(room_ix):
    manifest_path = "/om2/user/msaddler/spatial_audio_pipeline/assets/brir/eval/manifest_brir.pdpkl"
    index_room = room_ix
    h5_fn = get_room_str(room_ix)
    return dict(room_manifest=manifest_path, index_room=index_room, h5_fn=h5_fn)

# create dict of trial dicts
all_cond_dict = {ix:{'target_loc': (target_azim, elevation),
                      'distract_loc': [dist_az, elevation],
                      'snr': snr,
                      'test_room_meta': get_room_meta_dict(room_ix)} for ix, (dist_az, snr, room_ix) in
              enumerate(itertools.product(*[distractor_azims, snrs, room_ixs]))} 

n_all_cond_dict = len(all_cond_dict)
print(n_all_cond_dict)

# Assert keys contiguous
assert np.all(np.array(list(all_cond_dict.keys())) == np.arange(len(all_cond_dict)))

[  6.   3.   0.  -3.  -6.  -9. -12. -15. -18. -21.]
[0]
190


In [4]:
# # all_conds
# # all conditions 
with open('binaural_test_manifests/symmetric_distractor_front_back_thresholds_neg_21_to_6_dBSNR.pkl', 'wb') as f:
    pickle.dump(all_cond_dict, f)

In [9]:
# # all_conds
# # all conditions 
# with open('binaural_test_manifests/symmetric_distractor_conditions_neg_12_to_6_dBSNR.pkl', 'rb') as f:
#     cond_dict = pickle.load(f)

In [8]:
# First add conditions for confusion matrix 
target_azim = 0
elevation = 0 
distractor_azims = [0, 5, 10, 20, 30, 40, 60, 90] # Add 30, 60, 90 for comparison to our human experiments - test script handles l/r positioning 
# snrs = np.linspace(3,-15,num=7)
# create dict of trial dicts
all_cond_dict = {ix:{'target_loc': (target_azim, elevation),
                      'distract_loc': [dist_az, elevation],
                      'snr': 6} for ix, dist_az in
              enumerate(distractor_azims, )} 

n_all_cond_dict = len(all_cond_dict)
print(n_all_cond_dict)

# Assert keys contiguous
assert np.all(np.array(list(all_cond_dict.keys())) == np.arange(len(all_cond_dict)))

8


# sanity check old rooms 

In [2]:
room_manifest = pd.read_pickle('/om2/user/msaddler/spatial_audio_pipeline/assets/brir/mit_bldg46room1004/manifest_room.pdpkl')

In [6]:
room_ir_df = pd.read_pickle('/om2/user/msaddler/spatial_audio_pipeline/assets/brir/mit_bldg46room1004/manifest_brir.pdpkl')
elevations = room_ir_df[(room_ir_df.src_dist == 1.4) & (room_ir_df.index_room == 0)].src_elev.unique() # np.arange(-20, 41, 10)
azimuths = room_ir_df[(room_ir_df.src_dist == 1.4) & (room_ir_df.index_room == 0)].src_azim.unique() # np.arange(-20, 41, 10)
# remove 0 elevation
print(azimuths)
print(elevations)


[  0   5  10  15  20  25  30  35  40  45  50  55  60  65  70  75  80  85
  90  95 100 105 110 115 120 125 130 135 140 145 150 155 160 165 170 175
 180 185 190 195 200 205 210 215 220 225 230 235 240 245 250 255 260 265
 270 275 280 285 290 295 300 305 310 315 320 325 330 335 340 345 350 355]
[-20 -10   0  10  20  30  40]


In [7]:
# First add conditions for confusion matrix 
target_azim = 0
elevation = 0 
distractor_azims = [0, 5, 10, 20, 30, 40, 60, 90] # Add 30, 60, 90 for comparison to our human experiments - test script handles l/r positioning 
snrs = np.linspace(6,-15,num=8)
print(snrs)
room_ixs = [0]
print(room_ixs)

def get_room_str(room_ix):
    return f"/om2/user/msaddler/spatial_audio_pipeline/assets/brir/mit_bldg46room1004/room{room_ix:04}.hdf5"

def get_room_meta_dict(room_ix):
    manifest_path = "/om2/user/msaddler/spatial_audio_pipeline/assets/brir/mit_bldg46room1004/manifest_brir.pdpkl"
    index_room = room_ix
    h5_fn = get_room_str(room_ix)
    return dict(room_manifest=manifest_path, index_room=index_room, h5_fn=h5_fn)

# create dict of trial dicts
all_cond_dict = {ix:{'target_loc': (target_azim, elevation),
                      'distract_loc': [dist_az, elevation],
                      'snr': snr,
                      'test_room_meta': get_room_meta_dict(room_ix)} for ix, (dist_az, snr, room_ix) in
              enumerate(itertools.product(*[distractor_azims, snrs, room_ixs]))} 

n_all_cond_dict = len(all_cond_dict)
print(n_all_cond_dict)

# Assert keys contiguous
assert np.all(np.array(list(all_cond_dict.keys())) == np.arange(len(all_cond_dict)))

[  6.   3.   0.  -3.  -6.  -9. -12. -15.]
[0]
64


In [10]:
# all_conds
# all conditions 
with open('binaural_test_manifests/symmetric_distractor_conditions_neg_15_to_6_dBSNR_mit_room.pkl', 'wb') as f:
    pickle.dump(all_cond_dict, f)

## Simulate planned human experiment measuring thresholds at elevation and azimuth combinations 


Target at either -20 or 40 elevation

Elevation conditions:
* Distractors separated by 0, 20, and 40 degrees elevation relative to target at same azimuth (0 degrees)

Azimuth conditions:
* Distractors at 0, 10, and 30 degrees azimuth, symmetrically positioned relative to target at same target elevation

SNRs to measure 

-21 to 6 in 3dB steps 


In [12]:
np.linspace(6,-21,num=10)


array([  6.,   3.,   0.,  -3.,  -6.,  -9., -12., -15., -18., -21.])

In [16]:
target_azim = 0
elevations = [-20, 40]

distractor_azims = [0, 10, 30] 

distractor_elevation_deltas = [0, 20, 40]

snrs = np.linspace(6,-21,num=10)
room_ixs = [0]

def get_room_str(room_ix):
    return f"/om2/user/msaddler/spatial_audio_pipeline/assets/brir/mit_bldg46room1004/room{room_ix:04}.hdf5"

def get_room_meta_dict(room_ix):
    manifest_path = "/om2/user/msaddler/spatial_audio_pipeline/assets/brir/mit_bldg46room1004/manifest_brir.pdpkl"
    index_room = room_ix
    h5_fn = get_room_str(room_ix)
    return dict(room_manifest=manifest_path, index_room=index_room, h5_fn=h5_fn)

# create dict of trial dicts for azimuth experiment 

azim_cond_dict = {ix:{'target_loc': (0, elev),
                      'distract_loc': [dist_az, elev],
                      'snr': snr,
                      'symmetric_distractor': True,
                      'test_room_meta': get_room_meta_dict(room_ix)} for ix, (elev, dist_az, snr, room_ix) in
              enumerate(itertools.product(*[elevations, distractor_azims, snrs, room_ixs]))} 

n_azim_conds = len(azim_cond_dict)
# create dict of trial dicts for elevation experiment 
elev_cond_dict = {n_azim_conds + ix :{'target_loc': (0, elev),
                      'distract_loc': [0, elev - (np.sign(elev) * elev_delta)],
                      'snr': snr,
                      'symmetric_distractor': False,
                      'test_room_meta': get_room_meta_dict(room_ix)} for ix, (elev, elev_delta, snr, room_ix) in
              enumerate(itertools.product(*[elevations, distractor_elevation_deltas, snrs, room_ixs]))} 


all_cond_dict = {**azim_cond_dict, **elev_cond_dict}
# Assert keys contiguous
assert np.all(np.array(list(all_cond_dict.keys())) == np.arange(len(all_cond_dict)))

In [17]:
# all_conds
# all conditions 
with open('binaural_test_manifests/sim_2024_human_experiment.pkl', 'wb') as f:
    pickle.dump(all_cond_dict, f)

In [18]:
print(len(all_cond_dict))

120
